# 01 - Data Understanding

## Objective

Inspect both Central Water Commission (CWC) surface water quality CSV files independently to understand their structure, columns, data types as stored, and basic coverage before any cleaning, combining, or analysis.

## Purpose and Context

The project investigates surface water chemical quality across CWC monitoring stations in Uttar Pradesh, India. Two files are provided:

- Historical file: 1961 - 2020
- Recent file: 2021 - 2025

They may not share identical schemas, units, or data-quality characteristics. This notebook documents what is actually present, not what we assume.

## Datasets

- `data/raw/swq_manual_chemical_parameters_cwc_up_1961_2020.csv`
- `data/raw/swq_chemical_parameter_manual_cwc_up_2021_2025.csv`

Raw files are stored locally and excluded from Git (see `.gitignore`).

## Approach

Load each file as raw strings to prevent Pandas from silently converting values during inspection. Counts, dtypes, and date ranges will be documented from real output only.

### Imports

In [4]:
# Standard library
from pathlib import Path

# Data handling
import pandas as pd

# Display options: show more columns and wider text, for inspection only
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print("pandas version:", pd.__version__)

pandas version: 3.0.6


### Define file paths

In [17]:
# Project root is two levels up from this notebook:
# notebooks/01_data_understanding.ipynb -->  project root
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

HISTORICAL_FILE = RAW_DIR / "swq_manual_chemical_parameters_cwc_up_1961_2020.csv"
RECENT_FILE     = RAW_DIR / "swq_chemical_parameter_manual_cwc_up_2021_2025.csv"

# Sanity check: do both files exist?
print("Historical file exists:", HISTORICAL_FILE.exists())
print("Recent file exists:    ", RECENT_FILE.exists())

Historical file exists: True
Recent file exists:     True


### Load both files as strings

In [3]:
# Load both CSVs as strings. This preserves the raw file content
# and prevents Pandas from guessing data types during inspection.
# Empty cells still become NaN by default, so we can count missing values.

historical_df = pd.read_csv(HISTORICAL_FILE, dtype=str, low_memory=False)
recent_df     = pd.read_csv(RECENT_FILE,     dtype=str, low_memory=False)

print("Historical file shape (rows, columns):", historical_df.shape)
print("Recent file shape (rows, columns):    ", recent_df.shape)

Historical file shape (rows, columns): (30359, 52)
Recent file shape (rows, columns):     (12178, 52)


### Confirm strings and show root path

In [5]:
# Transparency: confirm the resolved project root and raw folder.
# These paths will appear in the notebook output for reproducibility.
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:     ", RAW_DIR)
print()

# Confirm every column in both DataFrames was read as string (object dtype).
# If we see anything other than object, our dtype=str instruction was bypassed.
print("Historical dtypes unique:", historical_df.dtypes.unique())
print("Recent dtypes unique:    ", recent_df.dtypes.unique())

PROJECT_ROOT: C:\Users\Himanshu\Documents\up-ganga-water-quality-analysis
RAW_DIR:      C:\Users\Himanshu\Documents\up-ganga-water-quality-analysis\data\raw

Historical dtypes unique: [<StringDtype(storage='python', na_value=nan)>]
Recent dtypes unique:     [<StringDtype(storage='python', na_value=nan)>]


### Compare column names

In [6]:
# Column names as ordered lists
hist_cols = historical_df.columns.tolist()
recent_cols = recent_df.columns.tolist()

# Are the two ordered lists identical?
print("Columns identical (order-sensitive):", hist_cols == recent_cols)
print()

# If they are not identical, identify what differs.
hist_set = set(hist_cols)
recent_set = set(recent_cols)

only_in_historical = hist_set - recent_set
only_in_recent = recent_set - hist_set

print("Columns only in historical file:", sorted(only_in_historical))
print("Columns only in recent file:    ", sorted(only_in_recent))
print()

# Order comparison: same set but different order?
if hist_set == recent_set and hist_cols != recent_cols:
    print("Same columns, different ORDER. Differences at these positions:")
    for i, (h, r) in enumerate(zip(hist_cols, recent_cols)):
        if h != r:
            print(f"  index {i}: historical='{h}'  recent='{r}'")

Columns identical (order-sensitive): True

Columns only in historical file: []
Columns only in recent file:     []



### Sample rows

In [18]:
# Show the first three rows of each file, transposed.
# Transposing lets us see all 52 columns as rows, so we can read
# the stored values column-by-column instead of squinting sideways.

print("=== HISTORICAL - first 3 rows (transposed) ===")
display(historical_df.head(3).T)

print()
print("=== RECENT - first 3 rows (transposed) ===")
display(recent_df.head(3).T)

=== HISTORICAL - first 3 rows (transposed) ===


,0,1,2
SlNo,1,2,3
Station,AGRA (J.B.),AGRA (J.B.),AGRA (J.B.)
Agency,CWC,CWC,CWC
State LGD Code,9,9,9
State,Uttar Pradesh,Uttar Pradesh,Uttar Pradesh
District LGD Code,118,118,118
District,AGRA,AGRA,AGRA
Tehsil,AGRA,AGRA,AGRA
Block,-,-,-
Village,-,-,-



=== RECENT - first 3 rows (transposed) ===


,0,1,2
SlNo,1,2,3
Station,AGRA (J.B.),AGRA (J.B.),AGRA (J.B.)
Agency,CWC,CWC,CWC
State LGD Code,9,9,9
State,Uttar Pradesh,Uttar Pradesh,Uttar Pradesh
District LGD Code,118,118,118
District,AGRA,AGRA,AGRA
Tehsil,AGRA,AGRA,AGRA
Block,-,-,-
Village,-,-,-


### Schema summary

In [19]:
# Build a per-column summary for each file.
# This is inspection only — we make no cleaning decisions yet.

def schema_summary(df):
    return pd.DataFrame({
        "dtype":     df.dtypes.astype(str),
        "non_null":  df.notna().sum(),
        "null":      df.isna().sum(),
        "null_pct":  (df.isna().mean() * 100).round(2),
        "nunique":   df.nunique(dropna=True),
    })

hist_summary   = schema_summary(historical_df)
recent_summary = schema_summary(recent_df)

print("=== HISTORICAL - schema summary ===")
display(hist_summary)

print()
print("=== RECENT - schema summary ===")
display(recent_summary)

=== HISTORICAL - schema summary ===


,dtype,non_null,null,null_pct,nunique
SlNo,str,30359,0,0.00,30359
Station,str,30359,0,0.00,99
Agency,str,30359,0,0.00,1
State LGD Code,str,30359,0,0.00,1
State,str,30359,0,0.00,1
District LGD Code,str,30359,0,0.00,53
District,str,30359,0,0.00,53
Tehsil,str,30359,0,0.00,68
Block,str,30359,0,0.00,7
Village,str,30359,0,0.00,7



=== RECENT - schema summary ===


,dtype,non_null,null,null_pct,nunique
SlNo,str,12178,0,0.00,12178
Station,str,12178,0,0.00,95
Agency,str,12178,0,0.00,1
State LGD Code,str,12178,0,0.00,1
State,str,12178,0,0.00,1
District LGD Code,str,12178,0,0.00,49
District,str,12178,0,0.00,49
Tehsil,str,12178,0,0.00,65
Block,str,12178,0,0.00,7
Village,str,12178,0,0.00,7


 ### Count hyphen(-) in all columns

In [20]:
# Count how many cells contain exactly the "-" placeholder in each file.
# This reveals hidden missingness that NaN counts did not capture.

def count_dash(df):
    # Compare each cell to "-" using .eq(), sum per column.
    # Using eq("-") on a StringDtype DataFrame returns a boolean DataFrame.
    return df.eq("-").sum()

hist_dash = count_dash(historical_df)
recent_dash = count_dash(recent_df)

# Only show columns that actually contain "-"
hist_dash_nonzero = hist_dash[hist_dash > 0].sort_values(ascending=False)
recent_dash_nonzero = recent_dash[recent_dash > 0].sort_values(ascending=False)

print("=== HISTORICAL - columns containing '-' ===")
print(hist_dash_nonzero)
print()
print("=== RECENT - columns containing '-' ===")
print(recent_dash_nonzero)

=== HISTORICAL - columns containing '-' ===
Block              28913
Village            28913
SubSubtributary    23966
Subtributary       16771
Tributary           4944
Local River           18
dtype: int64

=== RECENT - columns containing '-' ===
Block              11389
Village            11389
SubSubtributary    10160
Subtributary        6439
Tributary           2166
Tehsil               286
Local River          144
dtype: int64


### Inspect non-numeric tokens in a key numeric column

In [21]:
# Inspect distinct values in Iron(mg/L) that are NOT valid numbers.
# This reveals sentinel values, quality flags, or "<" detection-limit markers.

def non_numeric_values(series):
    # to_numeric with errors='coerce' converts valid numbers to float,
    # and turns anything invalid into NaN. Then we find which original
    # values became NaN — those are the non-numeric tokens.
    numeric = pd.to_numeric(series, errors="coerce")
    return series[numeric.isna() & series.notna()].unique()

print("Historical - non-numeric values in Iron(mg/L):")
print(non_numeric_values(historical_df["Iron(mg/L)"]))
print()
print("Recent - non-numeric values in Iron(mg/L):")
print(non_numeric_values(recent_df["Iron(mg/L)"]))

Historical - non-numeric values in Iron(mg/L):
<StringArray>
[]
Length: 0, dtype: str

Recent - non-numeric values in Iron(mg/L):
<StringArray>
[]
Length: 0, dtype: str


## Locate parameter columns

In [12]:
# Parameter columns run from "Amonia N (mgN/L)" to "Copper (mg/L)" inclusive.
# Locate them by name so the logic survives minor schema changes.

param_start = historical_df.columns.get_loc("Amonia N (mgN/L)")
param_end   = historical_df.columns.get_loc("Copper (mg/L)") + 1  # exclusive
param_cols  = historical_df.columns[param_start:param_end].tolist()

print(f"Number of parameter columns: {len(param_cols)}")
print()
for i, c in enumerate(param_cols, start=1):
    print(f"  {i:2d}. {c}")

Number of parameter columns: 33

   1. Amonia N (mgN/L)
   2. Boron (mg/L)
   3. Carbonate (mg/L)
   4. Calcium (mg/L)
   5. Chloride (mg/L)
   6. Dissolved oxygen (mg/L)
   7. Total Dissolved Solids (mg/L)
   8. Fluoride (mg/L)
   9. Bicarbonate (mg/L)
  10. Potassium (mg/L)
  11. Magnesium (mg/L)
  12. Sodium (mg/L)
  13. Nitrite N+Nitrate N (mgN/L)
  14. Potential of Hydrogen (pH)
  15. Sulphate (mg/L)
  16. Total Phosphorus (mgP/L)
  17. Arsenic (mg/L)
  18. Total Alkalinity (mg/L as CaCO3)
  19. Cadmium (mg/L)
  20. Chromium (mg/L)
  21. Iron(mg/L)
  22. Hardness Calcium (mgCaCO3/L)
  23. Mercury(mg/L)
  24. Hardness_Magnesium (mg/L as CaCO3)
  25. Total Hardness (mgCaCO3/L)
  26. Manganese (mg/L)
  27. Nitrate N (mgN/L)
  28. Nickel (mg/L)
  29. Lead (mg/L)
  30. Zinc (mg/L)
  31. Sodium Adsorption Ratio (%)
  32. Nitrate (mg/L)
  33. Copper (mg/L)


### Find non-numeric tokens per parameter column

In [22]:
# For every parameter column, find values that are:
#   - not missing (not NaN)
#   - not equal to "-" (already counted in Cell 9)
#   - not parseable as a number
# Report each affected column and the distinct offending tokens with counts.

def non_numeric_tokens(df, cols):
    report = {}
    for c in cols:
        s = df[c]
        numeric = pd.to_numeric(s, errors="coerce")
        # cells that became NaN during coercion but were not missing originally
        failed = s[numeric.isna() & s.notna()]
        # exclude the "-" placeholder; it is handled separately
        failed = failed[failed != "-"]
        if len(failed) > 0:
            report[c] = failed.value_counts().to_dict()
    return report

hist_tokens   = non_numeric_tokens(historical_df, param_cols)
recent_tokens = non_numeric_tokens(recent_df,     param_cols)

print("=== HISTORICAL - non-numeric tokens in parameter columns ===")
if not hist_tokens:
    print("None found. All non-empty parameter values are numeric.")
else:
    for c, tokens in hist_tokens.items():
        print(f"\n{c}")
        for tok, n in tokens.items():
            print(f"    {tok!r}  x{n}")

print()
print("=== RECENT - non-numeric tokens in parameter columns ===")
if not recent_tokens:
    print("None found. All non-empty parameter values are numeric.")
else:
    for c, tokens in recent_tokens.items():
        print(f"\n{c}")
        for tok, n in tokens.items():
            print(f"    {tok!r}  x{n}")

=== HISTORICAL - non-numeric tokens in parameter columns ===
None found. All non-empty parameter values are numeric.

=== RECENT - non-numeric tokens in parameter columns ===
None found. All non-empty parameter values are numeric.


 ### Date range per file

In [23]:
# Parse Data Acquisition Time explicitly with dayfirst=True.
# Sample values like "11-06-2007 08:00" and "01-01-2021 09:40" are DD-MM-YYYY.
# Pandas default is month-first; passing dayfirst=True is required here.
# We parse to datetime only for range inspection — the underlying DataFrame
# still holds the original string in this notebook.

def date_range(df, label):
    dt = pd.to_datetime(df["Data Acquisition Time"], dayfirst=True, errors="coerce")
    n_invalid = dt.isna().sum()
    print(f"=== {label} - Data Acquisition Time ===")
    print(f"  Rows:            {len(dt)}")
    print(f"  Unparseable:     {n_invalid}")
    print(f"  Earliest:        {dt.min()}")
    print(f"  Latest:          {dt.max()}")
    print(f"  Distinct dates:  {dt.dt.date.nunique()}")
    print()

date_range(historical_df, "HISTORICAL")
date_range(recent_df,     "RECENT")

=== HISTORICAL - Data Acquisition Time ===
  Rows:            30359
  Unparseable:     0
  Earliest:        1963-08-01 08:30:00
  Latest:          2020-12-21 14:40:00
  Distinct dates:  2437

=== RECENT - Data Acquisition Time ===
  Rows:            12178
  Unparseable:     0
  Earliest:        2021-01-01 07:00:00
  Latest:          2025-04-23 10:00:00
  Distinct dates:  332



### Station overlap between the two files

In [24]:
# Compare the set of station names between the two files.
# We use exact string matching for now. Case or whitespace differences
# will show up as "only in one file" — that would be a Data quality issue to fix.

hist_stations   = set(historical_df["Station"].dropna().unique())
recent_stations = set(recent_df["Station"].dropna().unique())

only_hist   = hist_stations - recent_stations
only_recent = recent_stations - hist_stations
in_both     = hist_stations & recent_stations

print(f"Historical stations:     {len(hist_stations)}")
print(f"Recent stations:         {len(recent_stations)}")
print(f"In both files:           {len(in_both)}")
print(f"Only in historical:      {len(only_hist)}")
print(f"Only in recent:          {len(only_recent)}")
print()
print("Stations only in historical file:")
for s in sorted(only_hist):
    print("   ", s)
print()
print("Stations only in recent file:")
for s in sorted(only_recent):
    print("   ", s)

Historical stations:     99
Recent stations:         95
In both files:           85
Only in historical:      14
Only in recent:          10

Stations only in historical file:
    B.K.Ghat
    Basantpur_CWC
    Bewar
    Bhinga
    Chandradeepghat
    Gagan Moradabad
    Jaunpur
    Kachhlabridge
    LUCKNOW
    Lucknow (Hanuman Setu)
    Mathura (Prayag ghat)
    NARHAN
    Rae bareli
    Rampur_GSM

Stations only in recent file:
    ARNOTA
    Basantpur
    KALPI
    Kachhla Bridge
    LALPUR
    Lucknow
    ORAI-2 Mangrayan
    Raebareli
    SITAMARHI
    Vrindavan-Yamuna ExpresswayLink Road Bridge U/S of Mathura


### Normalized station comparison

In [16]:
# Normalize station names to a comparison-safe form.
# We operate on copies, not the original DataFrame columns.
# Normalization rules:
#   - lowercase
#   - replace "_" with " "
#   - strip leading/trailing whitespace
#   - collapse repeated whitespace

import re

def normalize_station(name):
    s = name.lower()
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

hist_norm   = historical_df["Station"].dropna().map(normalize_station)
recent_norm = recent_df["Station"].dropna().map(normalize_station)

hist_set_norm   = set(hist_norm.unique())
recent_set_norm = set(recent_norm.unique())

only_hist_norm   = hist_set_norm - recent_set_norm
only_recent_norm = recent_set_norm - hist_set_norm
in_both_norm     = hist_set_norm & recent_set_norm

print("=== After normalization (lowercase, whitespace, underscores) ===")
print(f"Historical distinct:   {len(hist_set_norm)}")
print(f"Recent distinct:       {len(recent_set_norm)}")
print(f"In both:               {len(in_both_norm)}")
print(f"Only in historical:    {len(only_hist_norm)}")
print(f"Only in recent:        {len(only_recent_norm)}")
print()
print("Only in historical (normalized):")
for s in sorted(only_hist_norm):
    print("   ", s)
print()
print("Only in recent (normalized):")
for s in sorted(only_recent_norm):
    print("   ", s)

=== After normalization (lowercase, whitespace, underscores) ===
Historical distinct:   99
Recent distinct:       95
In both:               86
Only in historical:    13
Only in recent:        9

Only in historical (normalized):
    b.k.ghat
    basantpur cwc
    bewar
    bhinga
    chandradeepghat
    gagan moradabad
    jaunpur
    kachhlabridge
    lucknow (hanuman setu)
    mathura (prayag ghat)
    narhan
    rae bareli
    rampur gsm

Only in recent (normalized):
    arnota
    basantpur
    kachhla bridge
    kalpi
    lalpur
    orai-2 mangrayan
    raebareli
    sitamarhi
    vrindavan-yamuna expresswaylink road bridge u/s of mathura


## Results and Observations

All findings below are based on executed cells in this notebook.

### File shapes

- Historical file (`swq_manual_chemical_parameters_cwc_up_1961_2020.csv`): **30,359 rows × 52 columns**
- Recent file (`swq_chemical_parameter_manual_cwc_up_2021_2025.csv`): **12,178 rows × 52 columns**

### Schema

- Both files have **identical column names in identical order**.
- All 52 columns were loaded as Pandas `StringDtype`, confirming the raw content was preserved without silent type conversion.
- No columns are unique to either file.

### Constant and near-constant columns

- `River`: **1 distinct value** in both files (all records are `Ganga`).
- `Basin`: **1 distinct value** in both files (all records are `Ganga`).
- `Agency`: **1 distinct value** in both files.
- `State LGD Code` and `State`: **1 distinct value** in both files.
- These columns will be retained for documentation but will not be used for grouping.

### River-hierarchy variation

- `Tributary`: 12 distinct (historical), 11 (recent).
- `Subtributary`: 20 distinct (historical), 21 (recent).
- `SubSubtributary`: 7 distinct in both.
- `Local River`: 31 distinct (historical), 30 (recent).
- These columns vary meaningfully and can support station grouping.

### Missing data patterns

- `NaN` represents truly empty cells.
- `-` is a sentinel placeholder used in `Block`, `Village`, `Subtributary`, `SubSubtributary`, `Tributary`, `Local River`, and (in the recent file only) `Tehsil`.
- `Block` and `Village` are >93% `-` in both files - not usable for grouping.
- `Boron`, `Fluoride`, `Potassium`, `Nitrate (mg/L)`, `Hardness_Magnesium`, `Manganese`, and `Mercury` are essentially empty in one or both files.
- The two files have **complementary measurement coverage**: e.g. `Total Hardness` is 7% null in historical but 79% null in recent; `Dissolved oxygen` is 50% null historically and 5% null recently.

### Non-numeric tokens

- A systematic search across all 33 parameter columns in both files found **no non-numeric tokens** (no `<0.01`, `BDL`, flags, or stray text).
- Parameter columns contain only numbers, empty cells, and `NaN`.

### Date coverage

- Historical: **1963-08-01** to **2020-12-21** (2,437 distinct sampling dates).
- Recent: **2021-01-01** to **2025-04-23** (332 distinct sampling dates).
- **Zero unparseable dates** in either file when parsed with `dayfirst=True`.
- The two files are **temporally continuous** - no gap between 2020-12-21 and 2021-01-01.
- Note: the historical file's filename says `1961_2020`, but the earliest record is **1963-08-01**. The observed range is what we document.

### Station coverage

- Historical: **99** distinct stations.
- Recent: **95** distinct stations.
- Exact string match overlap: **85** stations.
- After normalizing case, whitespace, and underscores: **86** stations.
- Remaining "only in" entries are genuine typographic or naming variants (e.g. `Basantpur_CWC` vs `Basantpur`, `Kachhlabridge` vs `Kachhla Bridge`) that require a documented manual mapping in Cleaning & validation.



## Interpretation

### What the schema tells us

Both files were clearly produced by the same data platform. The identical 52-column schema, all-string dtype, and consistent date format support that. The integration question is therefore not one of schema harmonisation, but of temporal continuity, station identity, and measurement-program comparability.

### What the constant columns tell us

`River` and `Basin` being `Ganga` throughout is not a data error — it reflects the source's classification scheme. The meaningful variation is in `Tributary`, `Subtributary`, `SubSubtributary`, and `Local River`, which describe the hydrological position of each monitoring station within the Ganga system.

### What the missingness tells us

The complementary coverage pattern historical rich in hardness/SAR, recent rich in general chemistry and ammonia - means any naive pooled comparison across 1963–2025 would be **misleading**. Apparent trends could reflect changes in the monitoring programme rather than changes in water quality. This is a critical limitation to document before any temporal analysis.

### What the sentinel `-` tells us

In hierarchy columns, `-` likely means "not applicable at this classification level", not "missing". A monitoring station located on a tributary may have no subtributary classification; that is information, not absence of data. Treating `-` as missing would overstate missingness.

In `Block` and `Village`, however, `-` reflects genuinely unavailable metadata for over 93% of records. We cannot rely on those columns.

### What the station overlap tells us

The high overlap (86 of ~95–99 stations) means the two files cover essentially the same monitoring network. Cross-period comparison at the station level is therefore feasible but only for stations present in both files and only for parameters with sufficient coverage in both periods.

## Validation and Quality Checks

- File existence verified with `Path.exists()` before loading.
- Both files loaded with `dtype=str` and `low_memory=False`; dtypes confirmed as `StringDtype` (no silent conversion).
- Column identity verified using both set comparison (overlap) and list comparison (order-sensitive equality).
- Missingness computed with `df.isna()` (correctly handles both `NaN` and, separately, empty string).
- Sentinel `-` counted independently with `df.eq("-")` so it does not contaminate `NaN` counts.
- Non-numeric detection used `pd.to_numeric(errors="coerce")` and a comparison against the original to isolate genuinely unparseable cells, this approach avoids silently dropping values.
- Date parsing explicitly used `dayfirst=True` based on observed format; result checked by counting `NaT` (zero found).
- Station overlap computed with set arithmetic; total counts cross-checked (`len(only) + len(both) == len(distinct)`).
- Normalization applied to copies only the underlying DataFrames were not modified.

## Conclusion

This notebook established a verified understanding of both files without modifying either.

Key conclusions:

1. The two CSV files are schema-compatible and temporally continuous.
2. `River` and `Basin` are constant throughout both files; the useful hydrological variation is in the deeper hierarchy columns.
3. Parameter columns are clean - no sentinel tokens, no quality flags, no detection-limit markers.
4. Missingness patterns are structurally different between the two files, which will constrain cross-period comparisons.
5. `Block` and `Village` are effectively unusable for grouping.
6. Station overlap is high, but exact and normalized string matching undercount it; a manual mapping table is required during cleaning.
7. Actual date coverage differs from what the filenames imply (1963, not 1961).

The objective of this notebook to describe the files as they actually are has been achieved.

## Next Steps

The next notebook **Data Quality Audit** will investigate:

- Per-row missingness (how many parameters are typically reported per record).
- Duplicate records (identical `SlNo`? identical `Station + Data Acquisition Time`?).
- Latitude and longitude plausibility (all within Uttar Pradesh? any zeros? any swapped?).
- Suspicious numeric values (pH > 14, negative concentrations, dissolved oxygen above saturation, etc.).
- The non-`-` content in `Block` and `Village` (7 distinct values each).
- Sampling frequency over time.
- Decisions to carry into the Cleaning and Validation stage.